#  Automatic Albanian Text Summarization using mT5 & PEFT (LoRA)

This notebook contains the end-to-end training, optimization, decoding parameter tuning, baseline comparison, and qualitative evaluation pipeline for fine-tuning `google/mt5-small` using Low-Rank Adaptation (LoRA) on the Albanian Text Summarization dataset.

## 1. Environment Setup & Dependency Alignment
Upgrading required backend acceleration libraries (`torchao`) to ensure compatibility with recent `peft` and `transformers` versions.

In [1]:
!pip install -U torchao

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.7 MB/s  0:00:00


## 2. Base Model Loading & LoRA Parameter Injection
Loading `google/mt5-small` and injecting trainable rank decomposition matrices into the Query (`q`) and Value (`v`) attention modules.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "google/mt5-small"

# Load base sequence-to-sequence model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Configure Low-Rank Adaptation (LoRA)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                          # Rank dimension for adapter matrices
    lora_alpha=32,                 # Scaling hyperparameter
    lora_dropout=0.1,              # Dropout probability to prevent overfitting
    target_modules=["q", "v"],     # Target attention projection matrices
)

# Wrap base model with PEFT adapters
model = get_peft_model(model, lora_config)

# Print parameter distribution
model.print_trainable_parameters()

## 3. Dataset Ingestion & Tokenization Pipeline
Defining context length boundaries (`MAX_SOURCE_LENGTH = 640`, `MAX_TARGET_LENGTH = 96`) and processing raw source articles and target summaries into model-ready tensor inputs.

In [ ]:
MAX_SOURCE_LENGTH = 640
MAX_TARGET_LENGTH = 96

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "/kaggle/input/datasets/heldilami/albaniansummarization/train.csv",
    "validation": "/kaggle/input/datasets/heldilami/albaniansummarization/val.csv",
    "test": "/kaggle/input/datasets/heldilami/albaniansummarization/test.csv",
}
raw_datasets = load_dataset("csv", data_files=data_files)

def preprocess_function(examples):
    inputs = tokenizer(
        examples["source_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=examples["target_summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

## 4. Trainer Configuration & Fine-Tuning Execution
Setting up `Seq2SeqTrainer` with automatic evaluation per epoch, model checkpointing based on best validation ROUGE-L score, and bfloat16 mixed-precision acceleration.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import numpy as np
import evaluate

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where((preds >= 0) & (preds < tokenizer.vocab_size), preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
    )
    return {k: round(v * 100, 2) for k, v in result.items()}

# Configure Seq2Seq training hyperparameters
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/mt5-albanian-summarization",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    bf16=True,
    fp16=False,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# Execute LoRA fine-tuning loop
trainer.train()

In [ ]:
# Save fine-tuned LoRA weights and adapted tokenizer
trainer.save_model("/kaggle/working/mt5-shqip-LoRA")
tokenizer.save_pretrained("/kaggle/working/mt5-shqip-LoRA")

print("Model saved successfully")

In [ ]:
# Evaluate model performance on validation set
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

## 5. Model Merging & Inference Acceleration
Merging trained LoRA adapters into the base `mt5-small` weights (`merge_and_unload()`) for standalone GPU inference.

In [ ]:
!pip install --upgrade "torchao>=0.16.0"

from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/mt5-small"
LORA_PATH = "/kaggle/input/notebooks/heldilami/modeltrain/mt5-shqip-LoRA"

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

# Merge LoRA layers into base architecture for faster generation
model = model.merge_and_unload()
model.eval()
model.to("cuda")

print("Model successfully loaded and merged.")

## 6. Test Set Text Generation (Optimized Beam Decoding)
Generating summaries across the test set using beam search (`num_beams=4`), length penalties, ngram repetition blocking (`no_repeat_ngram_size=2`), and repetition penalties (`repetition_penalty=1.12`).

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm

def generate_summary(text):
    inputs = tokenizer(
        text, max_length=MAX_SOURCE_LENGTH, truncation=True, return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            length_penalty=1.2,
            no_repeat_ngram_size=2,
            repetition_penalty=1.12,  
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

predictions = []
for text in tqdm(test_df["source_text"].tolist()):
    predictions.append(generate_summary(text))

test_df["model_prediction"] = predictions
print("Summary generation completed.")

## 7. Model Evaluation & Quantitative Metrics (ROUGE)
Calculating ROUGE-1, ROUGE-2, ROUGE-L, and ROUGE-Lsum scores for the fine-tuned model predictions against ground truth targets.

In [ ]:
model_results = rouge.compute(
    predictions=test_df["model_prediction"].tolist(),
    references=test_df["target_summary"].astype(str).tolist(),
)
print("=== mT5 + LoRA (with Repetition Fix) ===")
for k, v in model_results.items():
    print(f"{k}: {v*100:.2f}")

## 8. Extractive Baseline Comparisons (Lead-N)
Evaluating heuristic Lead-N baselines (Lead-8, Lead-12, Lead-20) on the test set to quantify the comparative gain of generative abstraction over leading-sentence extraction.

In [ ]:
for n in [8, 12, 20]:
    lead_preds = test_df["source_text"].apply(lambda t: " ".join(str(t).split()[:n])).tolist()
    refs = test_df["target_summary"].astype(str).tolist()
    result = rouge.compute(predictions=lead_preds, references=refs)
    print(f"Lead-{n}: ROUGE-1={result['rouge1']*100:.2f}  ROUGE-2={result['rouge2']*100:.2f}  ROUGE-L={result['rougeL']*100:.2f}")

## 9. Qualitative Output Sampling
Sampling test instances to perform qualitative error analysis between source texts, ground truth target summaries, and generated model output.

In [ ]:
sample = test_df.sample(10, random_state=42)

for i, row in sample.iterrows():
    print("=" * 80)
    print(f"SOURCE (Start):\n{row['source_text'][:300]}...")
    print(f"\nGROUND TRUTH (Target):\n{row['target_summary']}")
    print(f"\nMODEL GENERATION:\n{row['model_prediction']}")
    print()

## 10. Result Export for Excel & Thesis Documentation
Exporting full predictions dataset to CSV using `utf-8-sig` encoding to preserve Albanian special characters (`ë`, `ç`) when opened in Microsoft Excel.

In [ ]:
test_df.to_csv("/kaggle/working/test_predictions_with_summaries.csv", index=False, encoding="utf-8-sig")
print("Saved successfully with UTF-8 encoding for Excel!")